In first example, we will design a neural network that can discover binding motifs in DNA based on the results of an assay that determines whether a longer DNA sequence binds to the protein or not. Here, the longer DNA sequences are our independent variables (or predictors),i.e. Xᵢ, while the positive or negative response of the assay is the dependent variable (or response), i.e. Yᵢ .  
loading data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

SEQUENCES_URL = 'https://raw.githubusercontent.com/abidlabs/deep-learning-genomics-primer/master/sequences.txt'

sequences = requests.get(SEQUENCES_URL).text.split('\n')
sequences = list(filter(None, sequences))  # This removes empty sequences.

# Let's print the first few sequences.
pd.DataFrame(sequences, index=np.arange(1, len(sequences)+1),
             columns=['MySequences']).head()
print(pd.DataFrame(sequences, index=np.arange(1, len(sequences)+1),
             columns=['MySequences']).head())
print(len(sequences))

                                         MySequences
1  CCGAGGGCTATGGTTTGGAAGTTAGAACCCTGGGGCTTCTCGCGGA...
2  GAGTTTATATGGCGCGAGCCTAGTGGTTTTTGTACTTGTTTGTCGC...
3  GATCAGTAGGGAAACAAACAGAGGGCCCAGCCACATCTAGCAGGTA...
4  GTCCACGACCGAACTCCCACCTTGACCGCAGAGGTACCACCAGAGC...
5  GGCGACCGAACTCCAACTAGAACCTGCATAACTGGCCTGGGAGATA...
2000


In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
# The LabelEncoder encodes a sequence of bases as a sequence of integers.
integer_encoder = LabelEncoder()
# The OneHotEncoder converts an array of integers to a sparse matrix where
# each row corresponds to one possible value of each feature.
one_hot_encoder = OneHotEncoder(categories=[range(4)])
input_features = []
np.set_printoptions(threshold=40)
for sequence in sequences:

  integer_encoded = integer_encoder.fit_transform(list(sequence))
  #print(integer_encoded)
  integer_encoded = np.array(integer_encoded).reshape(-1, 1)
  one_hot_encoded = one_hot_encoder.fit_transform(integer_encoded)
  if (len(input_features)) % 200 == 0:print('one_hot_encoded:',one_hot_encoded.toarray().T,type(one_hot_encoded))#
  input_features.append(one_hot_encoded.toarray())
  # print('one_hot_encoded.toarray\n',one_hot_encoded.toarray(),type(one_hot_encoded.toarray()))

one_hot_encoded: [[0. 0. 0. ... 1. 0. 0.]
 [1. 1. 0. ... 0. 1. 1.]
 [0. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 1. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 1. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 1.]
 [1. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 1. 0. 0.]
 [0. 1. 1. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 1. 1. ... 1. 1. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 1. ... 1. 0. 1.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]] <class 'scipy.sparse._csr.csr_matrix'>
one_hot_encoded: [[0. 0. 0. ... 0. 1. 0.]
 [0. 1. 0. ... 0

Then use above code encode DNA sequecne into : [[0. 1. 0. 0.],[]...,[]],4-dimensional vector as element for each base.  
np.stack() transfer the outermost list into array. 
Below code load the labels (response variables). The labels are structured as follows: a "1" indicates that a protein bound to the sequence, while a "0" indicates that the protein did not. In the latest OneHotEncoder() only take numeric 0, 1 . So use dtype=int in np.array() to transfer str to int.
note:  
label "0" is encoded to [1, 0]
label "1" is encoded to [0, 1]
Next, we choose a neural network architecture to train the model. In this tutorial, we choose a simple **1D convolutional neural network (CNN**), which is commonly used in deep learning for functional genomics applications.

A CNN learns to recognize patterns that are generally invariant across space, by trying to match the input sequence to a number of learnable "filters" of a fixed size. In our dataset, the filters will be motifs within the DNA sequences. The CNN may then learn to combine these filters to recognize a larger structure (e.g. the presence or absence of a transcription factor binding site).

We will use the deep learning library `Keras`. As of 2017, `Keras` has been integrated into `TensorFlow`,  which makes it very easy to construct neural networks. We only need to specify the **kinds of layers** we would like to include in our network, and the **dimensionality** of each layer. The CNN we generate in this example consists of the following layers:

- _Conv1D_: We define our convolutional layer to have** 32 filters of size 12 bases**.

- _MaxPooling1D_: After the convolution, we use a pooling layer to down-sample the output of the each of the 32 convolutional filters. Though not always required, this is a typical form of non-linear down-sampling used in CNNs.

- _Flatten_: This layer flattens the output of the max pooling layer, combining the results of the convolution and pooling layers across all 32 filters.

- _Dense_: The first Dense tensor creates a layer (dense_1) that compresses the representation of the flattened layer, resulting in smaller layer with 16 tensors, and the second Dense function converges the tensors into the output layer (dense_2) that consists of the two possible response values (0 or 1).

We can see the details of the architecture of the neural network we have created by running `model.summary()`, which prints the dimensionality and number of parameters for each layer in our network.

In [12]:
input_features = np.stack(input_features)
print(input_features[1].T)
LABELS_URL = 'https://raw.githubusercontent.com/abidlabs/deep-learning-genomics-primer/master/labels.txt'

labels = requests.get(LABELS_URL).text.split('\n')
labels = list(filter(None, labels))  # removes empty sequences
print(labels[:10],labels[-10:])
one_hot_encoder = OneHotEncoder(categories=[range(2)])

labels = np.array(labels, dtype=int).reshape(-1, 1)
print(labels,type(labels))
# print(len(one_hot_encoder.fit_transform(labels).toarray()))
input_labels = one_hot_encoder.fit_transform(labels).toarray()
print('Labels:\n',labels.T)
print('One-hot encoded labels:\n',input_labels.T)

from sklearn.model_selection import train_test_split
#split the data into training and test data sets.
train_features, test_features, train_labels, test_labels = train_test_split(
    input_features, input_labels, test_size=0.25, random_state=42)
from tensorflow.keras.layers import Conv1D, Dense, MaxPooling1D, Flatten
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(Conv1D(filters=32, kernel_size=12,
                 input_shape=(train_features.shape[1], 4)))
model.add(MaxPooling1D(pool_size=4))
model.add(Flatten())
model.add(Dense(16, activation='relu'))
model.add(Dense(2, activation='softmax'))

model.compile(loss='binary_crossentropy', optimizer='adam',
              metrics=['binary_accuracy'])
model.summary()

[[0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [1. 0. 1. ... 0. 0. 1.]
 [0. 0. 0. ... 1. 0. 0.]]
['0', '0', '0', '1', '1', '1', '1', '0', '0', '0'] ['1', '1', '1', '0', '0', '0', '1', '0', '1', '1']
[[0]
 [0]
 [0]
 ...
 [0]
 [1]
 [1]] <class 'numpy.ndarray'>
Labels:
 [[0 0 0 ... 0 1 1]]
One-hot encoded labels:
 [[1. 1. 1. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 1.]]


d:\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 39, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 9, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 288)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │         4,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,226 (24.32 KB)

 Trainable params: 6,226 (24.32 KB)

 Non-trainable params: 0 (0.00 B)

At each step, a cross-entropy loss is calculated over all the masked k-mers. We optimize DNABERT with AdamW using the following parameters: β_1=0.9,β_2=0.98,ϵ=1e-6  and weight decay as 0.01.  